In [3]:
# ══════════════════════════════════════════════════════════════════
#  🌍 Carbon & Energy Explorer — Pure Python (Colab)
#  Matches the dark UI screenshot exactly using matplotlib + ipywidgets
# ══════════════════════════════════════════════════════════════════

# ─────────────────────────────────────────────────────────────────
# CELL 1 — Install & Imports
# ─────────────────────────────────────────────────────────────────
# !pip install matplotlib ipywidgets --quiet

import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
from matplotlib import font_manager
import ipywidgets as widgets
from IPython.display import display, clear_output
import warnings
warnings.filterwarnings("ignore")

# Enable interactive widgets in Colab
from google.colab import output
output.enable_custom_widget_manager()

print("✅ All imports ready.")


# ─────────────────────────────────────────────────────────────────
# CELL 2 — Load & Prepare Data
# ─────────────────────────────────────────────────────────────────

BASE_PATH = "/content/"   # change if using Google Drive

df_ff = pd.read_csv(BASE_PATH + "merged_emissions_fossil_fuels.csv")
df_re = pd.read_csv(BASE_PATH + "analysis_renewables_vs_emissions.csv")

# Filter to countries only
ff = df_ff[df_ff["entity_type"] == "Country"][
    ["Country", "year", "total_fossil_fuel_consumption_tj", "co2_mt"]
].copy()

re = df_re[df_re["entity_type"] == "Country"][
    ["Country", "year", "renewables_qbtu", "renewable_share_pct", "total_co2_mt"]
].copy()
re.rename(columns={"total_co2_mt": "co2_mt_re"}, inplace=True)

# Merge
merged = pd.merge(ff, re, on=["Country", "year"], how="inner")
merged["fossil_qbtu"] = merged["total_fossil_fuel_consumption_tj"] / 1055.06  # TJ → qBtu
merged["renewable_share_pct"] = merged["renewable_share_pct"].clip(lower=0)
merged = merged.dropna(subset=["co2_mt", "fossil_qbtu", "renewables_qbtu", "renewable_share_pct"])

COUNTRIES = sorted(merged[merged["year"].between(1990, 2023)]["Country"].unique().tolist())
YEAR_MIN  = int(merged["year"].min())
YEAR_MAX  = int(merged["year"].max())

print(f"✅ {len(COUNTRIES)} countries · {YEAR_MIN}–{YEAR_MAX}")


# ─────────────────────────────────────────────────────────────────
# CELL 3 — Color constants (matching the HTML exactly)
# ─────────────────────────────────────────────────────────────────

BG        = "#0d0f0e"
SURFACE   = "#141715"
PANEL     = "#1b1e1c"
BORDER    = "#1f2320"
TEXT1     = "#e8ebe6"
TEXT2     = "#8a9088"
TEXT3     = "#525750"
CO2_C     = "#e06b3f"
FOSSIL_C  = "#a09b8c"
RENEW_C   = "#4caf7d"
CO2_FILL  = "#e06b3f22"
RENEW_FILL= "#4caf7d1a"


# ─────────────────────────────────────────────────────────────────
# CELL 4 — Chart drawing function
# ─────────────────────────────────────────────────────────────────

def get_country_data(country):
    return merged[merged["Country"] == country].sort_values("year")

def draw_chart(country, year, mode):
    """Draw the main dual/triple axis chart matching the UI."""
    df = get_country_data(country)
    if df.empty:
        return

    years       = df["year"].values
    co2         = df["co2_mt"].values
    fossil      = df["fossil_qbtu"].values / 1000   # → QBTU (thousands)
    renew_pct   = df["renewable_share_pct"].values
    renew_qbtu  = df["renewables_qbtu"].values

    fig, ax1 = plt.subplots(figsize=(11, 5.6))
    fig.patch.set_facecolor(BG)
    ax1.set_facecolor(BG)

    # ── Grid ──────────────────────────────────
    ax1.grid(axis="both", color="#1e211f", linewidth=0.6, linestyle="-", zorder=0)
    ax1.set_axisbelow(True)

    # ── Spines ────────────────────────────────
    for spine in ax1.spines.values():
        spine.set_edgecolor(BORDER)
        spine.set_linewidth(0.6)

    # ── CO₂ (always plotted) ──────────────────
    ax1.plot(years, co2, color=CO2_C, linewidth=2.2, zorder=4,
             marker="o", markersize=3, markerfacecolor=CO2_C, markeredgewidth=0)
    ax1.fill_between(years, co2, alpha=0.13, color=CO2_C, zorder=3)
    ax1.set_ylabel("CO₂ (Mt)", color=CO2_C, fontsize=9, labelpad=8)
    ax1.tick_params(axis="y", colors=CO2_C, labelsize=8)
    ax1.tick_params(axis="x", colors=TEXT3, labelsize=8)
    ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    ax1.set_xlim(years[0] - 0.5, years[-1] + 0.5)

    # Mark selected year with vertical dashed line
    ax1.axvline(x=year, color=TEXT2, linewidth=0.8, linestyle="--", alpha=0.45, zorder=5)

    legend_handles = [
        Line2D([0], [0], color=CO2_C, linewidth=2.2,
               marker="o", markersize=4, label="CO₂ emissions (Mt)")
    ]

    # ── Mode: CO₂ vs Fossil ───────────────────
    if mode in ("co2_vs_fossil", "all_three"):
        ax2 = ax1.twinx()
        ax2.set_facecolor(BG)
        ax2.plot(years, fossil, color=FOSSIL_C, linewidth=1.6, zorder=4,
                 marker="o", markersize=3, markerfacecolor=FOSSIL_C, markeredgewidth=0,
                 linestyle="--" if mode == "all_three" else "-")
        ax2.set_ylabel("Fossil (QBTU)", color=FOSSIL_C, fontsize=9, labelpad=8,
                       rotation=270, va="bottom")
        ax2.tick_params(axis="y", colors=FOSSIL_C, labelsize=8)
        ax2.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))
        for spine in ax2.spines.values():
            spine.set_edgecolor(BORDER); spine.set_linewidth(0.6)
        legend_handles.append(
            Line2D([0], [0], color=FOSSIL_C, linewidth=1.6,
                   marker="o", markersize=4,
                   linestyle="--" if mode == "all_three" else "-",
                   label="Fossil fuels (QBTU)")
        )

    # ── Mode: CO₂ vs Renewables ───────────────
    if mode in ("co2_vs_renew", "all_three"):
        ax3 = ax1.twinx()
        ax3.set_facecolor(BG)
        if mode == "all_three":
            ax3.spines["right"].set_position(("axes", 1.12))
        ax3.plot(years, renew_pct, color=RENEW_C, linewidth=1.6, zorder=4,
                 marker="o", markersize=3, markerfacecolor=RENEW_C, markeredgewidth=0,
                 linestyle=":" if mode == "all_three" else "-")
        if mode == "co2_vs_renew":
            ax3.fill_between(years, renew_pct, alpha=0.10, color=RENEW_C, zorder=2)
        ax3.set_ylabel("Renewable share (%)", color=RENEW_C, fontsize=9, labelpad=8,
                       rotation=270, va="bottom")
        ax3.tick_params(axis="y", colors=RENEW_C, labelsize=8)
        ax3.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.1f}%"))
        for spine in ax3.spines.values():
            spine.set_edgecolor(BORDER); spine.set_linewidth(0.6)
        legend_handles.append(
            Line2D([0], [0], color=RENEW_C, linewidth=1.6,
                   marker="o", markersize=4,
                   linestyle=":" if mode == "all_three" else "-",
                   label="Renewable share (%)")
        )

    # ── X axis ticks ──────────────────────────
    tick_years = [y for y in [1990, 1995, 2000, 2005, 2010, 2015, 2020, 2023]
                  if YEAR_MIN <= y <= YEAR_MAX]
    ax1.set_xticks(tick_years)
    ax1.set_xticklabels([str(y) for y in tick_years])

    # ── Legend (top-right pill style) ─────────
    leg = ax1.legend(
        handles=legend_handles,
        loc="upper left",
        frameon=True,
        framealpha=0.85,
        facecolor=PANEL,
        edgecolor=BORDER,
        fontsize=8.5,
        labelcolor=TEXT2,
        handlelength=1.8,
        handleheight=0.8,
        borderpad=0.7,
        labelspacing=0.5,
    )

    # ── Title block (inside chart, top-left) ──
    subtitle_map = {
        "co2_vs_fossil": "CO₂ emissions vs fossil fuel consumption",
        "co2_vs_renew":  "CO₂ emissions vs renewable energy share",
        "all_three":     "CO₂ emissions, fossil fuels & renewables",
    }

    fig.text(0.01, 0.97, country,
             color=TEXT1, fontsize=17, fontweight="bold", fontstyle="italic",
             ha="left", va="top", transform=fig.transFigure)
    fig.text(0.01, 0.90, subtitle_map[mode],
             color=TEXT3, fontsize=8, ha="left", va="top",
             transform=fig.transFigure, fontfamily="monospace")

    plt.tight_layout(rect=[0, 0, 1, 0.88])
    plt.show()
    plt.close(fig)


# ─────────────────────────────────────────────────────────────────
# CELL 5 — Sidebar stat cards (pure text widgets)
# ─────────────────────────────────────────────────────────────────

CARD_CSS = """
<style>
  @import url('https://fonts.googleapis.com/css2?family=DM+Mono:wght@300;400;500&family=DM+Serif+Display:ital@0;1&family=DM+Sans:wght@300;400;500&display=swap');
  body, .widget-area { background: #0d0f0e !important; }

  .explorer-header {
    background: #0d0f0e;
    border-bottom: 1px solid #1f2320;
    padding: 20px 28px 16px;
    display: flex;
    justify-content: space-between;
    align-items: flex-end;
    margin-bottom: 0;
  }
  .wm-title {
    font-family: 'DM Serif Display', serif;
    font-style: italic;
    font-size: 20px;
    color: #e8ebe6;
    letter-spacing: -0.02em;
  }
  .wm-sub {
    font-family: 'DM Mono', monospace;
    font-size: 9px;
    color: #525750;
    letter-spacing: 0.12em;
    text-transform: uppercase;
    margin-top: 2px;
  }
  .hdr-meta {
    font-family: 'DM Mono', monospace;
    font-size: 9px;
    color: #525750;
    text-align: right;
    line-height: 1.8;
    letter-spacing: 0.06em;
  }

  .sidebar-label {
    font-family: 'DM Mono', monospace;
    font-size: 9px;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    color: #525750;
    margin-bottom: 6px;
    margin-top: 14px;
  }
  .year-big {
    font-family: 'DM Serif Display', serif;
    font-style: italic;
    font-size: 38px;
    color: #e8ebe6;
    line-height: 1;
    margin: 4px 0 2px;
  }
  .year-range {
    font-family: 'DM Mono', monospace;
    font-size: 9px;
    color: #525750;
  }

  .stat-card {
    background: #1b1e1c;
    border: 1px solid #1f2320;
    border-radius: 6px;
    padding: 11px 14px;
    display: flex;
    align-items: stretch;
    gap: 10px;
    margin-bottom: 6px;
  }
  .stat-accent { width: 3px; border-radius: 2px; flex-shrink: 0; }
  .stat-lbl {
    font-family: 'DM Mono', monospace;
    font-size: 9px;
    color: #525750;
    letter-spacing: 0.1em;
    text-transform: uppercase;
  }
  .stat-val {
    font-family: 'DM Mono', monospace;
    font-size: 20px;
    font-weight: 500;
    color: #e8ebe6;
    margin-top: 2px;
  }
  .stat-unit {
    font-family: 'DM Mono', monospace;
    font-size: 9px;
    color: #525750;
    margin-top: 2px;
  }

  .mode-tab {
    display: flex;
    align-items: center;
    gap: 9px;
    padding: 9px 12px;
    border-radius: 6px;
    border: 1px solid transparent;
    font-family: 'DM Sans', sans-serif;
    font-size: 12px;
    color: #8a9088;
    margin-bottom: 3px;
    cursor: default;
  }
  .mode-tab.active {
    background: #1b1e1c;
    border-color: #1f2320;
    color: #e8ebe6;
  }
  .dot { width: 8px; height: 8px; border-radius: 50%; flex-shrink: 0; }
  .sidebar-divider {
    height: 1px;
    background: #1f2320;
    margin: 16px 0;
  }
  .snap-label {
    font-family: 'DM Mono', monospace;
    font-size: 9px;
    letter-spacing: 0.14em;
    text-transform: uppercase;
    color: #525750;
    margin-bottom: 10px;
  }
  .footer-bar {
    border-top: 1px solid #1f2320;
    padding: 10px 28px;
    display: flex;
    justify-content: space-between;
    background: #0d0f0e;
    margin-top: 4px;
  }
  .footer-txt {
    font-family: 'DM Mono', monospace;
    font-size: 9px;
    color: #525750;
    letter-spacing: 0.08em;
    text-transform: uppercase;
  }
</style>
"""

def make_sidebar_html(country, year, mode):
    df   = get_country_data(country)
    row  = df[df["year"] == year]
    if row.empty:
        row = df.iloc[[-1]]
    r = row.iloc[0]

    co2_val    = f"{r['co2_mt']:,.2f}"
    fossil_val = f"{r['fossil_qbtu']/1000:.3f}"
    renew_val  = f"{max(0, r['renewable_share_pct']):.1f}%"

    modes = [
        ("co2_vs_fossil", f"<span style='background:linear-gradient(135deg,#e06b3f,#a09b8c)'></span>", "CO₂ vs Fossil Fuels"),
        ("co2_vs_renew",  f"<span style='background:linear-gradient(135deg,#e06b3f,#4caf7d)'></span>",  "CO₂ vs Renewables"),
        ("all_three",     f"<span style='background:conic-gradient(#e06b3f 0 120deg,#4caf7d 120deg 240deg,#a09b8c 240deg 360deg)'></span>", "All Three Together"),
    ]
    mode_html = ""
    for m_id, _, m_label in modes:
        active_cls = "active" if m_id == mode else ""
        dot_style  = {
            "co2_vs_fossil": "background:linear-gradient(135deg,#e06b3f,#a09b8c)",
            "co2_vs_renew":  "background:linear-gradient(135deg,#e06b3f,#4caf7d)",
            "all_three":     "background:conic-gradient(#e06b3f 0 120deg,#4caf7d 120deg 240deg,#a09b8c 240deg 360deg)",
        }[m_id]
        mode_html += f"""
        <div class="mode-tab {active_cls}">
          <span class="dot" style="{dot_style}"></span>
          {m_label}
        </div>"""

    n_ctries = len(COUNTRIES)
    return f"""
    {CARD_CSS}
    <div style="background:#0d0f0e; padding: 0 0 8px 0; min-width:240px; max-width:260px;">

      <div class="sidebar-label">Country</div>
      <div style="font-family:'DM Mono',monospace;font-size:12px;color:#e8ebe6;
                  background:#1b1e1c;border:1px solid #1f2320;border-radius:6px;
                  padding:8px 12px;margin-bottom:0;">{country}</div>

      <div class="sidebar-label" style="margin-top:18px;">Year</div>
      <div class="year-big">{year}</div>
      <div class="year-range">{YEAR_MIN} – {YEAR_MAX}</div>

      <div class="sidebar-divider"></div>

      <div class="sidebar-label" style="margin-top:0;">View Mode</div>
      {mode_html}

      <div class="sidebar-divider"></div>

      <div class="snap-label">Snapshot · {year}</div>

      <div class="stat-card">
        <div class="stat-accent" style="background:#e06b3f;"></div>
        <div>
          <div class="stat-lbl">CO₂ Emissions</div>
          <div class="stat-val">{co2_val}</div>
          <div class="stat-unit">million tonnes</div>
        </div>
      </div>

      <div class="stat-card">
        <div class="stat-accent" style="background:#a09b8c;"></div>
        <div>
          <div class="stat-lbl">Fossil Fuel Use</div>
          <div class="stat-val">{fossil_val}</div>
          <div class="stat-unit">quadrillion BTU</div>
        </div>
      </div>

      <div class="stat-card">
        <div class="stat-accent" style="background:#4caf7d;"></div>
        <div>
          <div class="stat-lbl">Renewable Share</div>
          <div class="stat-val">{renew_val}</div>
          <div class="stat-unit">% of primary energy</div>
        </div>
      </div>

    </div>
    """


# ─────────────────────────────────────────────────────────────────
# CELL 6 — Widget layout & interactivity
# ─────────────────────────────────────────────────────────────────

# ── Widgets ──────────────────────────────────
w_country = widgets.Combobox(
    options=COUNTRIES,
    value="China",
    ensure_option=True,
    layout=widgets.Layout(width="238px"),
    style={"description_width": "0px"},
)

w_year = widgets.IntSlider(
    value=2010,
    min=YEAR_MIN, max=YEAR_MAX, step=1,
    continuous_update=True,
    readout=False,
    layout=widgets.Layout(width="238px"),
)

w_mode = widgets.RadioButtons(
    options=[
        ("CO₂ vs Fossil Fuels", "co2_vs_fossil"),
        ("CO₂ vs Renewables",   "co2_vs_renew"),
        ("All Three Together",  "all_three"),
    ],
    value="co2_vs_fossil",
    layout=widgets.Layout(width="238px"),
)

# ── Output areas ─────────────────────────────
out_sidebar = widgets.Output()
out_chart   = widgets.Output()

# ── Header HTML ──────────────────────────────
header_html = widgets.HTML(f"""
{CARD_CSS}
<div class="explorer-header">
  <div>
    <div class="wm-title">Carbon &amp; Energy Explorer</div>
    <div class="wm-sub">{YEAR_MIN} – {YEAR_MAX} · {len(COUNTRIES)} countries</div>
  </div>
  <div class="hdr-meta">Data: IEA / EIA / OWID<br>CO₂ in megatonnes · Energy in QBTU</div>
</div>
""")

footer_html = widgets.HTML(f"""
{CARD_CSS}
<div class="footer-bar">
  <span class="footer-txt">Sources: IEA Energy Statistics · EIA International Data · Our World in Data</span>
  <span class="footer-txt">Fossil fuel = coal + gas + oil · Renewables exclude nuclear</span>
</div>
""")

# ── Sidebar widget labels ─────────────────────
def make_label(text):
    return widgets.HTML(
        f'<span style="font-family:\'DM Mono\',monospace;font-size:9px;'
        f'letter-spacing:0.14em;text-transform:uppercase;color:#525750;">{text}</span>'
    )

sidebar_box = widgets.VBox([
    make_label("COUNTRY"),
    w_country,
    widgets.HTML('<div style="height:6px"></div>'),
    make_label("YEAR"),
    w_year,
    widgets.HTML('<div style="height:6px"></div>'),
    widgets.HTML('<hr style="border:none;border-top:1px solid #1f2320;margin:6px 0">'),
    make_label("VIEW MODE"),
    w_mode,
    widgets.HTML('<hr style="border:none;border-top:1px solid #1f2320;margin:6px 0">'),
    out_sidebar,
], layout=widgets.Layout(
    width="260px",
    min_width="260px",
    background_color="#0d0f0e",
    padding="20px 18px 20px 18px",
    border_right="1px solid #1f2320",
    overflow_y="auto",
))

# Style the radio buttons
w_mode.style = {"description_width": "0px", "button_color": "#0d0f0e"}

# Full layout
dashboard = widgets.VBox([
    header_html,
    widgets.HBox(
        [sidebar_box, out_chart],
        layout=widgets.Layout(
            flex="1",
            background_color="#0d0f0e",
            align_items="stretch",
        )
    ),
    footer_html,
], layout=widgets.Layout(
    background_color="#0d0f0e",
    border="1px solid #1f2320",
    width="100%",
))

# ── Update function ───────────────────────────
def update(change=None):
    country = w_country.value
    year    = w_year.value
    mode    = w_mode.value

    if country not in COUNTRIES:
        return

    with out_sidebar:
        clear_output(wait=True)
        display(widgets.HTML(make_sidebar_html(country, year, mode)))

    with out_chart:
        clear_output(wait=True)
        # Set matplotlib dark background globally
        with plt.rc_context({
            "figure.facecolor": BG,
            "axes.facecolor":   BG,
            "text.color":       TEXT1,
            "axes.labelcolor":  TEXT2,
            "xtick.color":      TEXT3,
            "ytick.color":      TEXT3,
            "axes.edgecolor":   BORDER,
            "grid.color":       "#1e211f",
            "grid.linewidth":   0.6,
            "font.size":        9,
        }):
            draw_chart(country, year, mode)

# ── Wire events ──────────────────────────────
w_country.observe(update, names="value")
w_year.observe(update,    names="value")
w_mode.observe(update,    names="value")

# ── Launch ───────────────────────────────────
display(dashboard)
update()


# ─────────────────────────────────────────────────────────────────
# CELL 7 — (Optional) Export chart as PNG
# ─────────────────────────────────────────────────────────────────

def export_chart(country="China", year=2023, mode="co2_vs_fossil",
                 filename="carbon_chart.png", dpi=180):
    """Save a high-res version of the chart."""
    df = get_country_data(country)
    if df.empty:
        print("No data for", country); return

    years     = df["year"].values
    co2       = df["co2_mt"].values
    fossil    = df["fossil_qbtu"].values / 1000
    renew_pct = df["renewable_share_pct"].values

    with plt.rc_context({
        "figure.facecolor": BG, "axes.facecolor": BG,
        "text.color": TEXT1, "axes.labelcolor": TEXT2,
        "xtick.color": TEXT3, "ytick.color": TEXT3,
        "axes.edgecolor": BORDER, "grid.color": "#1e211f",
        "grid.linewidth": 0.6, "font.size": 9,
    }):
        fig, ax1 = plt.subplots(figsize=(13, 6.5))
        fig.patch.set_facecolor(BG)
        ax1.set_facecolor(BG)
        ax1.grid(axis="both", color="#1e211f", linewidth=0.6, zorder=0)
        ax1.set_axisbelow(True)

        ax1.plot(years, co2, color=CO2_C, linewidth=2.2, zorder=4,
                 marker="o", markersize=3.5, markerfacecolor=CO2_C, markeredgewidth=0)
        ax1.fill_between(years, co2, alpha=0.13, color=CO2_C, zorder=3)
        ax1.axvline(x=year, color=TEXT2, linewidth=0.8, linestyle="--", alpha=0.45)
        ax1.set_ylabel("CO₂ (Mt)", color=CO2_C, fontsize=10)
        ax1.tick_params(axis="y", colors=CO2_C, labelsize=9)
        ax1.tick_params(axis="x", colors=TEXT3, labelsize=9)
        ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

        if mode in ("co2_vs_fossil", "all_three"):
            ax2 = ax1.twinx()
            ax2.set_facecolor(BG)
            ax2.plot(years, fossil, color=FOSSIL_C, linewidth=1.8, zorder=4,
                     marker="o", markersize=3.5, markerfacecolor=FOSSIL_C, markeredgewidth=0,
                     linestyle="--" if mode == "all_three" else "-")
            ax2.set_ylabel("Fossil (QBTU)", color=FOSSIL_C, fontsize=10, rotation=270, va="bottom")
            ax2.tick_params(axis="y", colors=FOSSIL_C, labelsize=9)

        if mode in ("co2_vs_renew", "all_three"):
            ax3 = ax1.twinx()
            ax3.set_facecolor(BG)
            if mode == "all_three":
                ax3.spines["right"].set_position(("axes", 1.12))
            ax3.plot(years, renew_pct, color=RENEW_C, linewidth=1.8, zorder=4,
                     marker="o", markersize=3.5, markerfacecolor=RENEW_C, markeredgewidth=0,
                     linestyle=":" if mode == "all_three" else "-")
            if mode == "co2_vs_renew":
                ax3.fill_between(years, renew_pct, alpha=0.10, color=RENEW_C, zorder=2)
            ax3.set_ylabel("Renewable share (%)", color=RENEW_C, fontsize=10, rotation=270, va="bottom")
            ax3.tick_params(axis="y", colors=RENEW_C, labelsize=9)

        tick_years = [y for y in range(1990, YEAR_MAX+1, 5) if YEAR_MIN<=y<=YEAR_MAX]
        ax1.set_xticks(tick_years)
        ax1.set_xticklabels([str(y) for y in tick_years])

        fig.text(0.01, 0.97, country, color=TEXT1, fontsize=20,
                 fontweight="bold", fontstyle="italic", ha="left", va="top",
                 transform=fig.transFigure)

        plt.tight_layout(rect=[0, 0, 1, 0.93])
        plt.savefig(filename, dpi=dpi, bbox_inches="tight", facecolor=BG)
        plt.close()
        print(f"✅ Saved → {filename}")

# Uncomment to export:
# export_chart("United States", 2020, "co2_vs_fossil", "us_co2_fossil.png")
# export_chart("Germany",       2023, "co2_vs_renew",  "germany_renew.png")
# export_chart("China",         2023, "all_three",     "china_all.png")

✅ All imports ready.
✅ 202 countries · 1990–2023


In [8]:
import os

# 1. Find the notebook file in /content/
notebooks = [f for f in os.listdir('/content/') if f.endswith('.ipynb')]

if not notebooks:
    print("❌ No .ipynb file found. Go to File > Save to ensure it exists in /content/")
else:
    # Pick the first notebook found (usually there is only one in Colab)
    target_nb = notebooks[0]
    output_html = target_nb.replace('.ipynb', '.html')

    print(f"Found: {target_nb}. Converting...")

    # 2. Run the conversion command
    # --no-input hides the code cells for a clean dashboard look
    # --template lab preserves the modern Jupyter look
    !jupyter nbconvert --to html --no-input --execute --template lab "{target_nb}"

    if os.path.exists(output_html):
        print(f"✅ SUCCESS! Download '{output_html}' from the file sidebar on the left.")
    else:
        print("❌ Conversion failed. Check the logs above.")

❌ No .ipynb file found. Go to File > Save to ensure it exists in /content/


In [4]:
!pip install nbconvert --quiet

In [6]:
# Replace 'Your_Notebook_Name.ipynb' with the actual name of your file
!jupyter nbconvert --to html --no-input --execute --template full "/content/carbon_emission.ipynb"

[NbConvertApp] WARNING | pattern '/content/carbon_emission.ipynb' matched no files
This application is used to convert notebook files (*.ipynb)
        to various other formats.


Options
The options below are convenience aliases to configurable class-options,
as listed in the "Equivalent to" description-line of the aliases.
To see all configurable class-options for some <cmd>, use:
    <cmd> --help-all

--debug
    set log level to logging.DEBUG (maximize logging output)
    Equivalent to: [--Application.log_level=10]
--show-config
    Show the application's configuration (human-readable format)
    Equivalent to: [--Application.show_config=True]
--show-config-json
    Show the application's configuration (json format)
    Equivalent to: [--Application.show_config_json=True]
--generate-config
    generate default config file
    Equivalent to: [--JupyterApp.generate_config=True]
-y
    Answer yes to any questions instead of prompting.
    Equivalent to: [--JupyterApp.answer_yes=True]
